In [ ]:
import os
import shutil
import random
from glob import glob
from sklearn.model_selection import train_test_split
import xml.etree.ElementTree as ET
from ultralytics import YOLO

In [ ]:

DATASET_DIR = r'C:\Users\User\Downloads\hard_hat_yolo'
DATA_YAML = os.path.join(DATASET_DIR, 'data.yaml')


In [ ]:
model = YOLO('yolo13s.pt')

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=416,
    batch=8,
    name='person_helmet_detector',
    patience=10,
    save=True,
    save_period=10,
    val=True,
    plots=True,
    device = 0
)

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO

MODEL_PATH = r"C:\Users\User\runs\detect\person_helmet_detector-3\weights\best.pt"
VAL_IMG_DIR = Path(r"C:\Users\User\Downloads\hard_hat_yolo\valid\images")
VAL_LBL_DIR = Path(r"C:\Users\User\Downloads\hard_hat_yolo\valid\labels")
IMG_SIZE = 640
CONF_THRESH = 0.25
IOU_THRESH = 0.5
OUTPUT_DIR = Path("error_images")

CLASS_NAMES = {0: "head", 1: "helmet"}

model = YOLO(MODEL_PATH)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def xywhn2xyxy(box, w, h):
    cx, cy, bw, bh = box
    cx *= w; cy *= h; bw *= w; bh *= h
    return [cx - bw/2, cy - bh/2, cx + bw/2, cy + bh/2]

def compute_iou(box1, box2):
    x1, y1, x2, y2 = max(box1[0], box2[0]), max(box1[1], box2[1]), min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    area1 = (box1[2]-box1[0])*(box1[3]-box1[1])
    area2 = (box2[2]-box2[0])*(box2[3]-box2[1])
    return inter / (area1 + area2 - inter + 1e-6)

total_gt = 0
total_pred = 0
total_fp = 0
total_fn = 0
processed = 0

for img_path in sorted(VAL_IMG_DIR.rglob("*.*")):
    label_path = VAL_LBL_DIR / (img_path.stem + ".txt")
    if not label_path.exists():
        continue

    img = cv2.imread(str(img_path))
    if img is None:
        continue
    h, w = img.shape[:2]

    gt_boxes = []
    if label_path.stat().st_size > 0:
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id, cx, cy, bw_, bh_ = map(float, parts)
                box = xywhn2xyxy([cx, cy, bw_, bh_], w, h)
                gt_boxes.append([int(cls_id), box])

    results = model.predict(img, imgsz=IMG_SIZE, conf=CONF_THRESH, verbose=False)
    preds = results[0].boxes
    pred_boxes = []
    if preds is not None:
        for *box, conf, cls_id in preds.data.tolist():
            pred_boxes.append([int(cls_id), box, conf])

    matched_gt = set()
    matched_pred = set()
    pred_boxes.sort(key=lambda x: x[2], reverse=True)

    for p_idx, (p_cls, p_box, p_conf) in enumerate(pred_boxes):
        best_iou = IOU_THRESH
        best_gt_idx = -1
        for g_idx, (g_cls, g_box) in enumerate(gt_boxes):
            if g_idx in matched_gt or g_cls != p_cls:
                continue
            iou = compute_iou(p_box, g_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = g_idx
        if best_gt_idx >= 0:
            matched_pred.add(p_idx)
            matched_gt.add(best_gt_idx)

    fp_boxes = [pred_boxes[i] for i in range(len(pred_boxes)) if i not in matched_pred]
    fn_boxes = [gt_boxes[i] for i in range(len(gt_boxes)) if i not in matched_gt]

    total_gt += len(gt_boxes)
    total_pred += len(pred_boxes)
    total_fp += len(fp_boxes)
    total_fn += len(fn_boxes)
    processed += 1

    if fp_boxes or fn_boxes:
        vis = img.copy()
        for g_cls, g_box in gt_boxes:
            cv2.rectangle(vis, (int(g_box[0]), int(g_box[1])), (int(g_box[2]), int(g_box[3])), (0,255,0), 2)
            cv2.putText(vis, CLASS_NAMES.get(g_cls, str(g_cls)), (int(g_box[0]), int(g_box[1])-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
        for p_cls, p_box, p_conf in fp_boxes:
            cv2.rectangle(vis, (int(p_box[0]), int(p_box[1])), (int(p_box[2]), int(p_box[3])), (0,0,255), 2)
            cv2.putText(vis, f"{CLASS_NAMES.get(p_cls, str(p_cls))} {p_conf:.2f}",
                        (int(p_box[0]), int(p_box[1])-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)
        for g_idx, (g_cls, g_box) in enumerate(gt_boxes):
            if g_idx not in matched_gt:
                cv2.rectangle(vis, (int(g_box[0]), int(g_box[1])), (int(g_box[2]), int(g_box[3])), (255,0,0), 3)
                cv2.putText(vis, "FN:" + CLASS_NAMES.get(g_cls, str(g_cls)),
                            (int(g_box[0]), int(g_box[1])-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 1)

        cv2.imwrite(str(OUTPUT_DIR / f"{img_path.stem}.jpg"), vis)

print(f"Обработано изображений: {processed}")
print(f"Всего GT-объектов: {total_gt}")
print(f"Всего предсказаний: {total_pred}")
print(f"Суммарно FP: {total_fp}, FN: {total_fn}")
if total_gt > 0:
    print(f"Recall: {1 - total_fn/total_gt:.3f}")
if total_pred > 0:
    print(f"Precision: {1 - total_fp/total_pred:.3f}")
print(f"Ошибочные кадры сохранены в: {OUTPUT_DIR}")

In [ ]:
import cv2
import torch
import pandas as pd
from datetime import datetime
from ultralytics import YOLO
import numpy as np
import os
import time
import json

try:
    import openpyxl
    OPENPYXL_AVAILABLE = True
except ImportError:
    OPENPYXL_AVAILABLE = False
    print("WARNING: openpyxl не установлен. Excel-отчёт не будет создан.")
    print("Для установки выполните: pip install openpyxl")

class SafetyMonitoringSystem:
    def __init__(self, model_path, conf_threshold=0.5, iou_threshold=0.45, device='cuda'):
        self.model = YOLO(model_path)
        self.model_path = model_path
        self.conf_threshold = conf_threshold
        self.iou_threshold = iou_threshold
        self.device = device if torch.cuda.is_available() and device == 'cuda' else 'cpu'
        self.overlap_threshold = 0.1

        self.total_people = 0
        self.violations_count = 0
        self.recorded_violators = set()
        self.violations_list = []

        self.screenshot_dir = 'violations_screenshots'
        self.violations_json = 'violations_history.json'
        self.runs_json = 'runs_history.json'

        self.frame_count = 0

    def log_violation(self, frame, bbox, confidence, track_id):
        """Логирует нарушение: сохраняет скриншот и добавляет запись в JSON."""
        if track_id is not None and track_id in self.recorded_violators:
            return

        os.makedirs(self.screenshot_dir, exist_ok=True)

        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
        filename = f"violation_{timestamp}_id{track_id}.jpg"
        filepath = os.path.join(self.screenshot_dir, filename)
        cv2.imwrite(filepath, frame)

        violation_entry = {
            'timestamp': timestamp,
            'frame_path': filepath,
            'confidence': float(confidence),
            'bbox': bbox.tolist() if isinstance(bbox, np.ndarray) else bbox,
            'track_id': int(track_id) if track_id is not None else None,
            'frame_number': self.frame_count
        }

        self.violations_list.append(violation_entry)
        self.violations_count += 1
        if track_id is not None:
            self.recorded_violators.add(track_id)

        try:
            with open(self.violations_json, 'r', encoding='utf-8') as f:
                all_violations = json.load(f)
        except (FileNotFoundError, json.JSONDecodeError):
            all_violations = []

        all_violations.append(violation_entry)
        with open(self.violations_json, 'w', encoding='utf-8') as f:
            json.dump(all_violations, f, ensure_ascii=False, indent=2)

    def get_overlap(self, box1, box2):
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        inter_area = max(0, x2 - x1) * max(0, y2 - y1)
        box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
        box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union_area = box1_area + box2_area - inter_area
        if union_area == 0:
            return 0
        return inter_area / union_area

    def process_frame(self, frame):
        """Обработка одного кадра: детекция, трекинг, проверка касок, аннотация."""
        self.frame_count += 1

        results = self.model.track(frame, conf=self.conf_threshold, iou=self.iou_threshold,
                                   device=self.device, persist=True)
        if results[0].boxes is None or results[0].boxes.id is None:
            results = self.model(frame, conf=self.conf_threshold, iou=self.iou_threshold, device=self.device)
            boxes = results[0].boxes.xyxy.cpu().numpy() if results[0].boxes is not None else []
            confs = results[0].boxes.conf.cpu().numpy() if results[0].boxes is not None else []
            cls_ids = results[0].boxes.cls.cpu().numpy().astype(int) if results[0].boxes is not None else []
            track_ids = [None] * len(boxes)
        else:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            confs = results[0].boxes.conf.cpu().numpy()
            cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
            track_ids = results[0].boxes.id.cpu().numpy().astype(int)

        heads = []
        helmets = []
        for bbox, conf, cls, tid in zip(boxes, confs, cls_ids, track_ids):
            if cls == 0:
                heads.append((bbox, conf, tid))
            elif cls == 1:
                helmets.append((bbox, conf))

        has_helmet = [False] * len(heads)
        for i, (head_bbox, _, _) in enumerate(heads):
            for helmet_bbox, _ in helmets:
                if self.get_overlap(head_bbox, helmet_bbox) > self.overlap_threshold:
                    has_helmet[i] = True
                    break

        annotated_frame = frame.copy()
        for i, (head_bbox, conf, tid) in enumerate(heads):
            x1, y1, x2, y2 = map(int, head_bbox)
            if has_helmet[i]:
                color = (0, 255, 0)
                label = f"Helmet {conf:.2f}"
            else:
                color = (0, 0, 255)
                label = f"NO HELMET! {conf:.2f}"
                self.log_violation(annotated_frame, head_bbox, conf, tid)
            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(annotated_frame, label, (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            if tid is not None:
                cv2.putText(annotated_frame, f"ID:{tid}", (x1, y1+20),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1)

        for helmet_bbox, conf in helmets:
            x1, y1, x2, y2 = map(int, helmet_bbox)
            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (255, 0, 0), 1)
            cv2.putText(annotated_frame, f"helmet {conf:.2f}", (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)

        self.total_people += len(heads)
        return annotated_frame, {
            'total_heads': len(heads),
            'violations_in_frame': sum(1 for h in has_helmet if not h),
            'total_violations_so_far': self.violations_count,
            'total_people_so_far': self.total_people,
            'frame_number': self.frame_count
        }

    def run_on_video(self, source, show=False, max_frames=None,
                     screenshot_dir='violations_screenshots',
                     json_violations='violations_history.json',
                     json_runs='runs_history.json',
                     excel_path='violations_report.xlsx',
                     process_every_n_frames=1):
        """
        Основной метод обработки видео.

        Args:
            source: путь к видео
            show: показывать ли окно с видео
            max_frames: максимальное количество кадров для обработки
            screenshot_dir: папка для скриншотов
            json_violations: путь к JSON с нарушениями
            json_runs: путь к JSON с запусками
            excel_path: путь к Excel-отчёту
            process_every_n_frames: обрабатывать каждый N-й кадр (для ускорения)
                                    Например: 1 - все кадры, 5 - каждый 5-й кадр,
                                    30 - каждый 30-й кадр (~1 кадр в секунду при 30fps)
        """
        self.screenshot_dir = screenshot_dir
        self.violations_json = json_violations
        self.runs_json = json_runs

        run_info = {
            'start_time': datetime.now().isoformat(),
            'video_source': source,
            'model_path': self.model_path,
            'conf_threshold': self.conf_threshold,
            'iou_threshold': self.iou_threshold,
            'device': self.device,
            'total_frames': 0,
            'total_people': 0,
            'total_violations': 0,
            'avg_fps': 0.0,
            'duration_sec': 0.0,
            'process_every_n_frames': process_every_n_frames
        }

        cap = cv2.VideoCapture(source)
        if not cap.isOpened():
            raise IOError(f"Не удалось открыть источник {source}")

        start_time = time.time()
        frame_count = 0
        self.frame_count = 0
        processed_frames = 0

        try:
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                frame_count += 1
                if max_frames and frame_count > max_frames:
                    break

                if frame_count % process_every_n_frames == 0:
                    annotated, stats = self.process_frame(frame)
                    processed_frames += 1

                    current_fps = processed_frames / (time.time() - start_time) if (time.time() - start_time) > 0 else 0
                    info_lines = [
                        f"People (total): {stats['total_people_so_far']}",
                        f"Violations (unique): {stats['total_violations_so_far']}",
                        f"Frame: {frame_count} (processed: {processed_frames})",
                        f"Current violations: {stats['violations_in_frame']}",
                        f"FPS: {current_fps:.1f}"
                    ]
                    for i, text in enumerate(info_lines):
                        cv2.putText(annotated, text, (10, 30 + i*25),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

                    if show:
                        cv2.imshow('Safety Monitoring', annotated)
                        if cv2.waitKey(1) & 0xFF == ord('q'):
                            break
                else:
                    if show:
                        display_frame = frame.copy()
                        cv2.putText(display_frame, f"Frame {frame_count} - skipped (processing every {process_every_n_frames})",
                                   (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                        cv2.imshow('Safety Monitoring', display_frame)
                        if cv2.waitKey(1) & 0xFF == ord('q'):
                            break

        except KeyboardInterrupt:
            print("Остановка пользователем")
        finally:
            cap.release()
            if show:
                cv2.destroyAllWindows()

        elapsed = time.time() - start_time
        avg_fps = processed_frames / elapsed if elapsed > 0 else 0

        run_info.update({
            'end_time': datetime.now().isoformat(),
            'total_frames': frame_count,
            'processed_frames': processed_frames,
            'total_people': self.total_people,
            'total_violations': self.violations_count,
            'avg_fps': avg_fps,
            'duration_sec': elapsed
        })

        self._save_run_to_history(run_info)

        if OPENPYXL_AVAILABLE:
            self.generate_excel_report(excel_path)
        else:
            print("Excel-отчёт не создан (openpyxl не установлен)")

        print(f"Обработано кадров: {processed_frames} из {frame_count} за {elapsed:.2f} сек (FPS: {avg_fps:.2f})")
        self.print_statistics()

    def _save_run_to_history(self, run_info):
        """Добавляет запись о запуске в JSON-файл истории."""
        try:
            with open(self.runs_json, 'r', encoding='utf-8') as f:
                runs = json.load(f)
        except (FileNotFoundError, json.JSONDecodeError):
            runs = []

        runs.append(run_info)
        with open(self.runs_json, 'w', encoding='utf-8') as f:
            json.dump(runs, f, ensure_ascii=False, indent=2)

    def generate_excel_report(self, excel_path='violations_report.xlsx'):
        """Создаёт Excel-отчёт: лист статистики запусков и лист нарушений."""
        if not OPENPYXL_AVAILABLE:
            print("Невозможно создать Excel-отчёт: openpyxl не установлен")
            return

        try:
            with open(self.runs_json, 'r', encoding='utf-8') as f:
                runs = json.load(f)
        except (FileNotFoundError, json.JSONDecodeError):
            runs = []

        try:
            with open(self.violations_json, 'r', encoding='utf-8') as f:
                violations = json.load(f)
        except (FileNotFoundError, json.JSONDecodeError):
            violations = []

        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            if runs:
                df_runs = pd.DataFrame(runs)
                cols_to_keep = ['start_time', 'end_time', 'video_source', 'total_frames',
                               'processed_frames', 'total_people', 'total_violations',
                               'avg_fps', 'duration_sec', 'process_every_n_frames']
                df_runs = df_runs[[col for col in cols_to_keep if col in df_runs.columns]]
                df_runs.to_excel(writer, sheet_name='Статистика запусков', index=False)
            else:
                pd.DataFrame({'Информация': ['Нет данных о запусках']}).to_excel(
                    writer, sheet_name='Статистика запусков', index=False)

            if violations:
                df_violations = pd.DataFrame(violations)
                df_violations['bbox'] = df_violations['bbox'].apply(lambda x: str(x))
                df_violations.to_excel(writer, sheet_name='Нарушения', index=False)
            else:
                pd.DataFrame({'Сообщение': ['Нет записей о нарушениях']}).to_excel(
                    writer, sheet_name='Нарушения', index=False)

        print(f"Excel-отчёт сохранён в {excel_path}")

    def print_statistics(self):
        print("\n=== СТАТИСТИКА ТЕКУЩЕГО ЗАПУСКА ===")
        print(f"Всего обнаружено людей (суммарно по кадрам): {self.total_people}")
        print(f"Уникальных нарушителей (без каски): {self.violations_count}")
        print(f"Сохранено скриншотов нарушений: {len(self.violations_list)}")
        print(f"История нарушений сохранена в: {self.violations_json}")
        print(f"История запусков сохранена в: {self.runs_json}")
        print("====================================\n")


if __name__ == "__main__":
    MODEL_PATH = "C:/Users/User/runs/detect/person_helmet_detector-3/weights/best.pt"
    VIDEO_PATH = "C:/Users/User/Downloads/Process_of_Constructing_a_Concrete_Modular_House_in_Just_2_Weeks.mp4"
    SCREENSHOT_DIR = "C:/Users/User/Downloads/violations_screenshots"

    print("Загрузка модели...")
    system = SafetyMonitoringSystem(
        model_path=MODEL_PATH,
        conf_threshold=0.5,
        device='cuda'
    )
    print("Модель загружена!")

    system.run_on_video(
        source=VIDEO_PATH,
        show=True,
        max_frames=None,
        screenshot_dir=SCREENSHOT_DIR,
        json_violations='violations_history.json',
        json_runs='runs_history.json',
        excel_path='violations_report.xlsx',
        process_every_n_frames=30
    )